# DABench Data Explorer

Three sections:
1. **Full dataset overview** — distributions across all 257 tasks
2. **Set comparison** — how well do our dev/eval subsets represent the full set?
3. **Dev set deep dive** — question-by-question inspection with CSV previews

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR   = Path('../data/dabench')
TABLES_DIR = DATA_DIR / 'tables'

# Load questions and labels
questions = [json.loads(l) for l in (DATA_DIR / 'questions.jsonl').read_text().splitlines()]
labels    = {l['id']: l for l in [json.loads(l) for l in (DATA_DIR / 'labels.jsonl').read_text().splitlines()]}

# Enrich each question with label info
for q in questions:
    q['n_answers'] = len(labels[q['id']]['common_answers'])
    q['answers']   = labels[q['id']]['common_answers']

df = pd.DataFrame(questions)

# Our chosen subsets
DEV_IDS  = [0, 9, 18, 5, 11, 62, 7, 23, 28, 39]
EVAL_IDS = [24, 32, 55, 27, 66, 69, 70, 77, 118, 124]

df['split'] = df['id'].apply(lambda x: 'dev' if x in DEV_IDS else ('eval' if x in EVAL_IDS else 'unused'))

print(f'Loaded {len(df)} questions across {df["file_name"].nunique()} CSV files')
df[['id','level','file_name','n_answers','split']].head()

---
## 1. Full Dataset Overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('DABench — Full Dataset (257 tasks)', fontsize=15, fontweight='bold')

level_order = ['easy', 'medium', 'hard']
level_colors = ['#4CAF50', '#FF9800', '#F44336']

# 1a — Difficulty distribution
ax = axes[0, 0]
level_counts = df['level'].value_counts()[level_order]
bars = ax.bar(level_counts.index, level_counts.values, color=level_colors)
for bar, val in zip(bars, level_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', fontsize=11)
ax.set_title('Difficulty Distribution')
ax.set_ylabel('Number of tasks')
ax.set_ylim(0, max(level_counts.values) * 1.2)

# 1b — Top concepts
ax = axes[0, 1]
all_concepts = [c for concepts in df['concepts'] for c in concepts]
concept_counts = Counter(all_concepts).most_common(8)
names, counts = zip(*concept_counts)
ax.barh(list(reversed(names)), list(reversed(counts)), color='steelblue')
ax.set_title('Top Concepts (multi-label)')
ax.set_xlabel('Number of tasks')

# 1c — Number of answers per task
ax = axes[1, 0]
answer_counts = df['n_answers'].value_counts().sort_index()
ax.bar(answer_counts.index, answer_counts.values, color='mediumpurple')
ax.set_title('Answers Required per Task')
ax.set_xlabel('Number of answers')
ax.set_ylabel('Number of tasks')
ax.set_xticks(answer_counts.index)

# 1d — Top CSV files
ax = axes[1, 1]
file_counts = df['file_name'].value_counts().head(10)
ax.barh(list(reversed(file_counts.index)), list(reversed(file_counts.values)), color='teal')
ax.set_title('Most Used CSV Files (top 10)')
ax.set_xlabel('Number of tasks')

plt.tight_layout()
plt.show()

---
## 2. Set Comparison — Full vs Dev vs Eval

Are our 10-task dev and eval subsets representative of the full dataset?

In [ ]:
full  = df
dev   = df[df['split'] == 'dev']
eval_ = df[df['split'] == 'eval']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Difficulty Distribution: Full (257) vs Dev (10) vs Eval (10)',
             fontsize=13, fontweight='bold')

for ax, (name, subset) in zip(axes, [('Full', full), ('Dev', dev), ('Eval', eval_)]):
    counts = subset['level'].value_counts().reindex(level_order, fill_value=0)
    pcts   = counts / counts.sum() * 100
    bars = ax.bar(pcts.index, pcts.values, color=level_colors, edgecolor='white', linewidth=1.5)
    for bar, pct, cnt in zip(bars, pcts.values, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.0f}%\n(n={cnt})', ha='center', fontsize=9)
    ax.set_title(f'{name} set', fontsize=12, fontweight='bold')
    ax.set_ylabel('% of tasks')
    ax.set_ylim(0, 65)

plt.tight_layout()
plt.show()

# Summary table
print('\n=== Summary Table ===')
for name, subset in [('Full', full), ('Dev', dev), ('Eval', eval_)]:
    print(f'\n{name} ({len(subset)} tasks):')
    print(f'  Difficulty : {dict(subset["level"].value_counts()[level_order])}')
    print(f'  Unique CSVs: {subset["file_name"].nunique()}')
    print(f'  Avg answers: {subset["n_answers"].mean():.1f}')

In [ ]:
# Concept coverage comparison
def concept_pcts(subset):
    all_c = [c for concepts in subset['concepts'] for c in concepts]
    total = len(subset)
    return {c: round(count / total * 100, 1) for c, count in Counter(all_c).most_common()}

full_pcts  = concept_pcts(full)
dev_pcts   = concept_pcts(dev)
eval_pcts  = concept_pcts(eval_)

top_concepts = list(full_pcts.keys())[:8]
comparison = pd.DataFrame({
    'Full %': [full_pcts.get(c, 0) for c in top_concepts],
    'Dev %':  [dev_pcts.get(c, 0)  for c in top_concepts],
    'Eval %': [eval_pcts.get(c, 0) for c in top_concepts],
}, index=top_concepts)

ax = comparison.plot(kind='barh', figsize=(12, 5),
                     color=['#90CAF9', '#A5D6A7', '#FFCC80'], edgecolor='white')
ax.set_title('Concept Coverage — % of tasks containing this concept', fontweight='bold')
ax.set_xlabel('% of tasks')
plt.tight_layout()
plt.show()

comparison

---
## 3. Dev Set Deep Dive

Full details for each dev task — question, constraints, expected answer format, ground truth, and a CSV preview.

In [ ]:
from IPython.display import display, Markdown

for _, row in dev.iterrows():
    display(Markdown(
        f"---\n"
        f"### Task {row['id']} — {row['level'].upper()}  \n"
        f"**Concepts:** {', '.join(row['concepts'])}  \n"
        f"**File:** `{row['file_name']}`  \n\n"
        f"**Question:** {row['question']}  \n\n"
        f"**Constraints:** {row['constraints']}  \n\n"
        f"**Expected format:** `{row['format']}`  \n\n"
        f"**Ground truth:** `{row['answers']}`"
    ))

    # CSV preview
    csv_path = TABLES_DIR / row['file_name']
    if csv_path.exists():
        try:
            csv_df = pd.read_csv(csv_path, nrows=5)
            size_kb = csv_path.stat().st_size // 1024
            display(Markdown(f"**CSV preview** ({size_kb} KB, first 5 rows):"))
            display(csv_df)
        except Exception as e:
            display(Markdown(f"_Could not read CSV: {e}_"))
    else:
        display(Markdown(f"_CSV not found at {csv_path}_"))